In [1]:
import hotspot
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mplscience
import pickle

In [2]:
!pwd
%cd /home/jovyan/work/COVID/parameter_tuning/hotspot/Cell
!pwd

/home/jovyan/work/COVID/parameter_tuning/hotspot
/home/jovyan/work/COVID/parameter_tuning/hotspot/Cell
/home/jovyan/work/COVID/parameter_tuning/hotspot/Cell


In [3]:
adata = sc.read_h5ad("../../../COVID_macrophage_scVI.h5ad")
adata

AnnData object with n_obs × n_vars = 56861 × 26571
    obs: 'PatientID', 'Sample type', 'CoVID-19 severity', 'datasets', 'batch', 'celltype', 'majorType', 'sampleID', 'City', 'Age', 'Sex', 'Sample time', 'Sampling day (Days after symptom onset)', 'SARS-CoV-2', 'Single cell sequencing platform', 'BCR single cell sequencing', 'TCR single cell sequencing', 'Outcome', 'Comorbidities', 'COVID-19-related medication and anti-microbials', 'Leukocytes [G/L]', 'Neutrophils [G/L]', 'Lymphocytes [G/L]', 'Unpublished', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type', '_scvi_batch', '_scvi_labels', 'leiden_scVI'
    var: 'gene_ids-0', 'feature_types-0', 'genome-0-0', 'genome-1-0', 'genome-10-0', 'genome-11-0', 'genome-2-0', 'genome-3-0', 'genome-4-0', 'genome-5-0', 'genome-6-0', 'genome-7-0', 'genome-8-0', 'genome-9-0', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_c

Ctrl

In [ ]:
adata_ctrl = adata[adata.obs["CoVID-19 severity"] == "control"]
adata_ctrl

In [ ]:
adata_mild = adata[adata.obs["CoVID-19 severity"] == "mild/moderate"]
adata_mild

In [4]:
adata_severe = adata[adata.obs["CoVID-19 severity"] == "severe/critical"]
adata_severe

View of AnnData object with n_obs × n_vars = 50866 × 26571
    obs: 'PatientID', 'Sample type', 'CoVID-19 severity', 'datasets', 'batch', 'celltype', 'majorType', 'sampleID', 'City', 'Age', 'Sex', 'Sample time', 'Sampling day (Days after symptom onset)', 'SARS-CoV-2', 'Single cell sequencing platform', 'BCR single cell sequencing', 'TCR single cell sequencing', 'Outcome', 'Comorbidities', 'COVID-19-related medication and anti-microbials', 'Leukocytes [G/L]', 'Neutrophils [G/L]', 'Lymphocytes [G/L]', 'Unpublished', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type', '_scvi_batch', '_scvi_labels', 'leiden_scVI'
    var: 'gene_ids-0', 'feature_types-0', 'genome-0-0', 'genome-1-0', 'genome-10-0', 'genome-11-0', 'genome-2-0', 'genome-3-0', 'genome-4-0', 'genome-5-0', 'genome-6-0', 'genome-7-0', 'genome-8-0', 'genome-9-0', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 

In [5]:
import gc
del adata
gc.collect()

2732

In [ ]:
hs = hotspot.Hotspot(
    adata_ctrl,
    layer_key="scvi_log",
    model='normal',
    latent_obsm_key="X_scVI",
    # umi_counts_obs_key="total_counts"
)

In [ ]:
hs.create_knn_graph(weighted_graph=False, n_neighbors=30)

In [ ]:
hs_results = hs.compute_autocorrelations()
hs_results.head(15)

In [ ]:
# Select the genes with significant lineage autocorrelation
hs_genes = hs_results.index

# Compute pair-wise local correlations between these genes
lcz = hs.compute_local_correlations(hs_genes, jobs=15)

In [ ]:
# Save the Hotspot object to a file
with open("hotspot_object_ctrl.pkl", "wb") as f:
    pickle.dump(hs, f)

In [ ]:
del hs
del adata_ctrl

Mild

In [ ]:
hs = hotspot.Hotspot(
    adata_mild,
    layer_key="scvi_log",
    model='normal',
    latent_obsm_key="X_scVI",
    # umi_counts_obs_key="total_counts"
)

In [ ]:
hs.create_knn_graph(weighted_graph=False, n_neighbors=30)

In [ ]:
hs_results = hs.compute_autocorrelations()

In [ ]:
# Select the genes with significant lineage autocorrelation
hs_genes = hs_results.index

# Compute pair-wise local correlations between these genes
lcz = hs.compute_local_correlations(hs_genes, jobs=15)

In [ ]:
# Save the Hotspot object to a file
with open("hotspot_object_mild.pkl", "wb") as f:
    pickle.dump(hs, f)

In [ ]:
del hs
del adata_mild

Severe

In [6]:
len(adata_severe)*0.25

12716.5

In [7]:
adata_severe_20 = sc.pp.subsample(adata_severe, n_obs=12717, copy=True, random_state=1234)

In [8]:
adata_severe_20

AnnData object with n_obs × n_vars = 12717 × 26571
    obs: 'PatientID', 'Sample type', 'CoVID-19 severity', 'datasets', 'batch', 'celltype', 'majorType', 'sampleID', 'City', 'Age', 'Sex', 'Sample time', 'Sampling day (Days after symptom onset)', 'SARS-CoV-2', 'Single cell sequencing platform', 'BCR single cell sequencing', 'TCR single cell sequencing', 'Outcome', 'Comorbidities', 'COVID-19-related medication and anti-microbials', 'Leukocytes [G/L]', 'Neutrophils [G/L]', 'Lymphocytes [G/L]', 'Unpublished', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type', '_scvi_batch', '_scvi_labels', 'leiden_scVI'
    var: 'gene_ids-0', 'feature_types-0', 'genome-0-0', 'genome-1-0', 'genome-10-0', 'genome-11-0', 'genome-2-0', 'genome-3-0', 'genome-4-0', 'genome-5-0', 'genome-6-0', 'genome-7-0', 'genome-8-0', 'genome-9-0', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_c

In [9]:
del adata_severe
gc.collect()

988

In [10]:
hs = hotspot.Hotspot(
    adata_severe_20,
    layer_key="scvi_log",
    model='normal',
    latent_obsm_key="X_scVI",
    # umi_counts_obs_key="total_counts"
)

In [11]:
del adata_severe_20
gc.collect()

1463

In [12]:
hs.create_knn_graph(weighted_graph=False, n_neighbors=30)

In [13]:
hs_results = hs.compute_autocorrelations()

100% 26571/26571 [00:58<00:00, 456.47it/s]


In [ ]:
# Select the genes with significant lineage autocorrelation/
hs_genes = hs_results.index

# Compute pair-wise local correlations between these genes
lcz = hs.compute_local_correlations(hs_genes, jobs=15)

Computing pair-wise local correlation on 26571 features...


100% 26571/26571 [00:12<00:00, 2108.71it/s]
  2% 7125720/352995735 [13:26<10:43:34, 8956.94it/s]

In [ ]:
# Save the Hotspot object to a file
with open("hotspot_object_severe.pkl", "wb") as f:
    pickle.dump(hs, f)

In [ ]:
del hs